In [1]:
import ipywidgets as widgets
from IPython.display import clear_output, display

# модель нечіткої бази знань

class Term:

  def __init__(self, name, term_type, a, b, c, d=None):
    self.name = name
    self.term_type = term_type
    self.a = float(a)
    self.b = float(b)
    self.c = float(c)
    self.d = float(d) if d is not None else float(c)

  def membership(self, x):
    """Обчислення ступеня належності для трикутної або трапецієподібної функції."""
    if self.term_type == 'triangular':

      if x <= self.a or x >= self.c:
        return 0.0
      if x == self.b:
        return 1.0
      if x < self.b:
        return (x - self.a) / (self.b - self.a) if self.b != self.a else 1.0
      return (self.c - x) / (self.c - self.b) if self.c != self.b else 1.0

    elif self.term_type == 'trapezoidal':

      if x <= self.a or x >= self.d:
        return 0.0
      if self.b <= x <= self.c:
        return 1.0
      if x < self.b:
        return (x - self.a) / (self.b - self.a) if self.b != self.a else 1.0
      return (self.d - x) / (self.d - self.c) if self.d != self.c else 1.0

    return 0.0


class FuzzyVariable:

  def __init__(self, name, minimum, maximum):
    self.name = name
    self.minimum = float(minimum)
    self.maximum = float(maximum)
    self.terms = {}

  def add_term(self, term):
    self.terms[term.name] = term


class FuzzyRule:

  def __init__(self, room_temp_term, out_temp_term, heater_term):
    self.room_temp_term = room_temp_term
    self.out_temp_term = out_temp_term
    self.heater_term = heater_term

# створення БД для розумного термостата

def create_thermostat_db():

  room_temp = FuzzyVariable('Кімнатна температура', 0, 30)
  room_temp.add_term(
      Term('Холодно', 'trapezoidal', 0, 0, 15, 18)
  )
  room_temp.add_term(Term('Комфортно', 'triangular', 16, 21, 24))
  room_temp.add_term(Term('Тепло', 'trapezoidal', 22, 25, 30, 30))

  out_temp = FuzzyVariable('Зовнішня температура', -20, 20)
  out_temp.add_term(Term('Мороз', 'trapezoidal', -20, -20, -5, 0))
  out_temp.add_term(Term('Прохолодно', 'triangular', -5, 5, 12))
  out_temp.add_term(Term('Тепло', 'trapezoidal', 8, 15, 20, 20))

  heater = FuzzyVariable('Потужність обігрівача', 0, 100)
  heater.add_term(Term('Вимкнено', 'triangular', 0, 0, 20))
  heater.add_term(Term('Слабка', 'triangular', 10, 35, 60))
  heater.add_term(
      Term('Максимальна', 'trapezoidal', 50, 75, 100, 100)
  )

  rules = [
      FuzzyRule('Холодно', 'Мороз', 'Максимальна'),
      FuzzyRule('Холодно', 'Прохолодно', 'Максимальна'),
      FuzzyRule('Холодно', 'Тепло', 'Слабка'),
      FuzzyRule('Комфортно', 'Мороз', 'Слабка'),
      FuzzyRule('Комфортно', 'Прохолодно', 'Слабка'),
      FuzzyRule('Комфортно', 'Тепло', 'Вимкнено'),
      FuzzyRule('Тепло', 'Мороз', 'Вимкнено'),
      FuzzyRule('Тепло', 'Прохолодно', 'Вимкнено'),
      FuzzyRule('Тепло', 'Тепло', 'Вимкнено'),
  ]

  return {
      'Кімнатна температура': room_temp,
      'Зовнішня температура': out_temp,
      'Потужність обігрівача': heater,
  }, rules


variables, rules = create_thermostat_db()

# інтерактивний графічний інтерфейс
out = widgets.Output()

# елементи вибору змінной, терма та типу функції
var_select = widgets.Dropdown(
    options=list(variables.keys()), description='Змінна:'
)
term_select = widgets.Dropdown(description='Терм:')
type_select = widgets.Dropdown(
    options=[('Трикутна', 'triangular'), ('Трапецієподібна', 'trapezoidal')],
    description='Тип ФН:',
)

# поля параметрів
a_float = widgets.FloatText(value=0, description='a:')
b_float = widgets.FloatText(value=0, description='b:')
c_float = widgets.FloatText(value=0, description='c:')
d_float = widgets.FloatText(value=0, description='d:')

btn_apply = widgets.Button(
    description='Оновити терм', button_style='primary', icon='check'
)

# вхідні дані для симуляції розумного термостата
room_in = widgets.FloatText(value=17.5, description='Кімната (°C):')
out_in = widgets.FloatText(value=-8.0, description='Зовні (°C):')
btn_run = widgets.Button(
    description='Розрахувати потужність',
    button_style='success',
    icon='calculator',
)


def update_terms_options(change=None):
  var_name = var_select.value
  term_select.options = list(variables[var_name].terms.keys())
  fill_params()


def fill_params(change=None):
  var_name = var_select.value
  t_name = term_select.value
  if t_name in variables[var_name].terms:
    term = variables[var_name].terms[t_name]
    type_select.value = term.term_type
    a_float.value = term.a
    b_float.value = term.b
    c_float.value = term.c
    d_float.value = term.d

    # динамічно ховається або показується параметр d залежно від типу ФН
    d_float.layout.display = (
        'block' if term.term_type == 'trapezoidal' else 'none'
    )


def on_type_change(change=None):
  d_float.layout.display = (
      'block' if type_select.value == 'trapezoidal' else 'none'
  )


var_select.observe(update_terms_options, names='value')
term_select.observe(fill_params, names='value')
type_select.observe(on_type_change, names='value')


def on_apply_clicked(b):
  var_name = var_select.value
  t_name = term_select.value
  term = variables[var_name].terms[t_name]

  term.term_type = type_select.value
  term.a = a_float.value
  term.b = b_float.value
  term.c = c_float.value
  term.d = (
      d_float.value if type_select.value == 'trapezoidal' else c_float.value
  )

  with out:
    clear_output()
    print(f"Оновлено параметри терма '{t_name}' для змінної '{var_name}':")
    print(
        f"  - Тип: {'Трапецієподібна' if term.term_type == 'trapezoidal' else 'Трикутна'}"
    )
    if term.term_type == 'trapezoidal':
      print(
          f'  - Параметри: a={term.a}, b={term.b}, c={term.c}, d={term.d}'
      )
    else:
      print(f'  - Параметри: a={term.a}, b={term.b}, c={term.c}')


def on_run_clicked(b):
  r_val = room_in.value
  o_val = out_in.value

  # фазифікація
  r_m = {
      k: v.membership(r_val)
      for k, v in variables['Кімнатна температура'].terms.items()
  }
  o_m = {
      k: v.membership(o_val)
      for k, v in variables['Зовнішня температура'].terms.items()
  }

  # агрегування
  aggregated = {
      k: 0.0 for k in variables['Потужність обігрівача'].terms
  }
  for rule in rules:
    act = min(
        r_m.get(rule.room_temp_term, 0.0), o_m.get(rule.out_temp_term, 0.0)
    )
    aggregated[rule.heater_term] = max(
        aggregated[rule.heater_term], act
    )

  # дефазифікація методом центру ваги
  heater_var = variables['Потужність обігрівача']
  num, den = 0.0, 0.0
  steps = 200

  for i in range(steps + 1):
    x = (
        heater_var.minimum
        + i * (heater_var.maximum - heater_var.minimum) / steps
    )
    deg = 0.0
    for t_name, act in aggregated.items():
      term = heater_var.terms[t_name]
      deg = max(deg, min(act, term.membership(x)))
    num += x * deg
    den += deg

  result = 0.0 if den == 0 else num / den

  verify_msg = (
      "Успішно! Зміна вхідних даних чи параметрів трапеції відображається у"
      " підсумку."
      if result >= 0
      else "Помилка розрахунку."
  )

  with out:
    clear_output()
    print('РЕЗУЛЬТАТ РОБОТИ РОЗУМНОГО ТЕРМОСТАТА')
    print(f'Кімнатна температура: {r_val} °C')
    print(f'Зовнішня температура: {o_val} °C\n')
    print('Ступені активації термостата:')
    for t_name, act in aggregated.items():
      print(f'  - {t_name}: {act:.2f}')
    print(f'\nРозрахована потужність обігрівача: {result:.2f} %')
    print(f'\n[Перевірка успішності кроку]: {verify_msg}')


btn_apply.on_click(on_apply_clicked)
btn_run.on_click(on_run_clicked)

# Ініціалізація початкових значень у формі
update_terms_options()

# Відображення інтерфейсу в блокноті
display(
    widgets.VBox([
        widgets.HTML('<h2>Редактор термів (Підтримка трапецій та трикутників)</h2>'),
        widgets.HBox([var_select, term_select, type_select]),
        widgets.HBox([a_float, b_float, c_float, d_float, btn_apply]),
        widgets.HTML('<h2>Симуляція розумного термостата</h2>'),
        widgets.HBox([room_in, out_in, btn_run]),
        out,
    ])
)